In [1]:
import numpy as np
import pandas as pd
import requests

In [2]:
df = pd.read_csv("../../data/merged_final_transformed.csv")

In [3]:
df.head()

,year,StateAbbr,County name,CountyFIPS,BPHIGH,CASTHMA,COPD,MHLTH,PHLTH,SLEEP,...,median_household_income,pct_male,pct_female,pct_less_than_hs,pct_bachelors_plus,pct_graduate_degree,pct_white,pct_black,pct_asian,pct_hispanic
0,2013,AK,ANCHORAGE MUNICIPALITY,2020,28.691667,NaN,NaN,NaN,NaN,NaN,...,77454.0,50.877431,49.122569,4.749405,13.285259,7.373060,66.225101,5.872909,8.224240,7.984433
1,2013,AL,JEFFERSON COUNTY,1073,41.893750,NaN,NaN,NaN,NaN,NaN,...,45429.0,47.338099,52.661901,8.438817,12.332967,7.788755,53.203847,42.299621,1.474751,3.803496
2,2013,AL,MADISON COUNTY,1089,37.654348,NaN,NaN,NaN,NaN,NaN,...,58434.0,49.020128,50.979872,6.620805,15.974758,9.615096,69.198506,24.052771,2.489986,4.586491
3,2013,AL,MOBILE COUNTY,1097,44.833333,NaN,NaN,NaN,NaN,NaN,...,43028.0,47.976708,52.023292,10.492560,8.919669,4.602989,60.570249,34.773759,1.899862,2.473692
4,2013,AL,MONTGOMERY COUNTY,1101,39.792727,NaN,NaN,NaN,NaN,NaN,...,44790.0,47.572277,52.427723,9.241418,12.436021,7.702419,40.095642,55.189099,2.122893,3.481090


In [4]:
old_age = df[df["median_age"] > 70]
print(f"Rows with median_age > 70: {len(old_age)}")
old_age

Rows with median_age > 70: 0


,year,StateAbbr,County name,CountyFIPS,BPHIGH,CASTHMA,COPD,MHLTH,PHLTH,SLEEP,...,median_household_income,pct_male,pct_female,pct_less_than_hs,pct_bachelors_plus,pct_graduate_degree,pct_white,pct_black,pct_asian,pct_hispanic


In [5]:
# Define parameters
api_key = "71166a96655234262760b0975365279ba3b63e04"
years = range(2013, 2024)

In [7]:
all_data = []

for year in years:
    # Profile variables (B-series)
    profile_vars = "NAME,B01001_001E,B01001_002E,B01001_026E,B01002_001E,B02001_002E,B02001_003E,B02001_005E,B03002_012E,B19013_001E,B06009_002E,B06009_005E,B06009_006E"
    url_profile = f"https://api.census.gov/data/{year}/acs/acs5?get={profile_vars}&for=county:*&key={api_key}"

    r1 = requests.get(url_profile)

    if r1.status_code == 200:
        df1 = pd.DataFrame(r1.json()[1:], columns=r1.json()[0])
        df1["year"] = year
        all_data.append(df1)
    else:
        print(f"{year} — profile: {r1.status_code}, subject: {r2.status_code}")
        if r1.status_code != 200:
            print(f"  Profile error: {r1.text}")

final_df = pd.concat(all_data, ignore_index=True)

In [8]:
final_df.head()

,NAME,B01001_001E,B01001_002E,B01001_026E,B01002_001E,B02001_002E,B02001_003E,B02001_005E,B03002_012E,B19013_001E,B06009_002E,B06009_005E,B06009_006E,state,county,year
0,"Escambia County, Florida",300795,149023,151772,37.4,208609,66610,8388,14856,43918,23995,30575,16318,12,033,2013
1,"Hernando County, Florida",173119,82934,90185,48.0,155440,9170,1816,18416,41024,17203,13491,6239,12,053,2013
2,"Hillsborough County, Florida",1257913,613671,644242,36.3,913889,212004,43742,317679,49596,109534,158926,86612,12,057,2013
3,"Okaloosa County, Florida",185852,93520,92332,37.4,149824,16841,5825,13839,54684,11452,20595,13739,12,091,2013
4,"Taylor County, Florida",22660,12770,9890,41.4,17133,4862,57,831,36356,3852,1132,770,12,123,2013


In [9]:
final_df["B01001_001E"] = pd.to_numeric(final_df["B01001_001E"], errors="coerce")
final_df["B01001_002E"] = pd.to_numeric(final_df["B01001_002E"], errors="coerce")
final_df["B01001_026E"] = pd.to_numeric(final_df["B01001_026E"], errors="coerce")
final_df["B06009_005E"] = pd.to_numeric(final_df["B06009_005E"], errors="coerce")
final_df["B06009_006E"] = pd.to_numeric(final_df["B06009_006E"], errors="coerce")
final_df["B06009_002E"] = pd.to_numeric(final_df["B06009_002E"], errors="coerce")
final_df["B02001_002E"] = pd.to_numeric(final_df["B02001_002E"], errors="coerce")
final_df["B02001_003E"] = pd.to_numeric(final_df["B02001_003E"], errors="coerce")
final_df["B02001_005E"] = pd.to_numeric(final_df["B02001_005E"], errors="coerce")
final_df["B03002_012E"] = pd.to_numeric(final_df["B03002_012E"], errors="coerce")

final_df["pct_male"]   = final_df["B01001_002E"] / final_df["B01001_001E"] * 100
final_df["pct_female"] = final_df["B01001_026E"] / final_df["B01001_001E"] * 100
final_df["pct_less_than_hs"] = final_df["B06009_002E"] / final_df["B01001_001E"] * 100
final_df["pct_bachelors_plus"] = final_df["B06009_005E"] / final_df["B01001_001E"] * 100
final_df["pct_graduate_degree"] = final_df["B06009_006E"] / final_df["B01001_001E"] * 100
final_df["pct_white"] = final_df["B02001_002E"] / final_df["B01001_001E"] * 100
final_df["pct_black"] = final_df["B02001_003E"] / final_df["B01001_001E"] * 100
final_df["pct_asian"] = final_df["B02001_005E"] / final_df["B01001_001E"] * 100
final_df["pct_hispanic"] = final_df["B03002_012E"] / final_df["B01001_001E"] * 100
final_df = final_df.drop(columns=["B01001_002E", "B01001_026E", "B06009_006E", "B06009_005E", "B06009_002E",
                                  "B02001_002E", "B02001_003E", "B02001_005E", "B03002_012E"])

In [10]:
# update column names
final_df = final_df.rename(columns={
    "B01001_001E":  "total_population",
    "B01002_001E":  "median_age",
    "B19013_001E":  "median_household_income"
})

print(final_df.columns.tolist())
final_df.head()

['NAME', 'total_population', 'median_age', 'median_household_income', 'state', 'county', 'year', 'pct_male', 'pct_female', 'pct_less_than_hs', 'pct_bachelors_plus', 'pct_graduate_degree', 'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic']


,NAME,total_population,median_age,median_household_income,state,county,year,pct_male,pct_female,pct_less_than_hs,pct_bachelors_plus,pct_graduate_degree,pct_white,pct_black,pct_asian,pct_hispanic
0,"Escambia County, Florida",300795,37.4,43918,12,033,2013,49.543044,50.456956,7.977194,10.164730,5.424957,69.352549,22.144650,2.788610,4.938912
1,"Hernando County, Florida",173119,48.0,41024,12,053,2013,47.905776,52.094224,9.937095,7.792905,3.603879,89.787949,5.296934,1.048989,10.637769
2,"Hillsborough County, Florida",1257913,36.3,49596,12,057,2013,48.784852,51.215148,8.707597,12.634101,6.885373,72.651209,16.853630,3.477347,25.254449
3,"Okaloosa County, Florida",185852,37.4,54684,12,091,2013,50.319609,49.680391,6.161892,11.081398,7.392441,80.614683,9.061511,3.134214,7.446248
4,"Taylor County, Florida",22660,41.4,36356,12,123,2013,56.354810,43.645190,16.999117,4.995587,3.398058,75.609003,21.456311,0.251545,3.667255


In [11]:
# Convert all numeric columns
numeric_cols = [col for col in final_df.columns if col not in ["NAME", "state", "county"]]
final_df[numeric_cols] = final_df[numeric_cols].apply(pd.to_numeric, errors="coerce")

print(final_df.dtypes)

NAME                           str
total_population             int64
median_age                 float64
median_household_income    float64
state                          str
county                         str
year                         int64
pct_male                   float64
pct_female                 float64
pct_less_than_hs           float64
pct_bachelors_plus         float64
pct_graduate_degree        float64
pct_white                  float64
pct_black                  float64
pct_asian                  float64
pct_hispanic               float64
dtype: object


In [12]:
old_age_new = final_df[final_df["median_age"] > 70]
print(f"Rows with median_age > 70: {len(old_age_new)}")
old_age_new

Rows with median_age > 70: 0


,NAME,total_population,median_age,median_household_income,state,county,year,pct_male,pct_female,pct_less_than_hs,pct_bachelors_plus,pct_graduate_degree,pct_white,pct_black,pct_asian,pct_hispanic


In [13]:
# Save copy BEFORE merge
df_original = df.copy()

In [14]:
# Build CountyFIPS in final_df to match df
final_df["CountyFIPS"] = final_df["state"].astype(str).str.zfill(2) + final_df["county"].astype(str).str.zfill(3)
final_df["CountyFIPS"] = final_df["CountyFIPS"].astype(int)

# Columns to replace
replace_cols = ["total_population", "median_age", "median_household_income",
                "pct_male", "pct_female", "pct_less_than_hs", "pct_bachelors_plus",
                "pct_graduate_degree", "pct_white", "pct_black", "pct_asian", "pct_hispanic"]

# Drop old columns from df
df = df.drop(columns=[c for c in replace_cols if c in df.columns])

# Merge in new columns from final_df
df = df.merge(
    final_df[["CountyFIPS", "year"] + replace_cols],
    on=["CountyFIPS", "year"],
    how="left"
)

print(f"Shape after merge: {df.shape}")
print(f"Missing values in new columns:\n{df[replace_cols].isnull().sum()}")
df.head()

Shape after merge: (6646, 41)
Missing values in new columns:
total_population           0
median_age                 0
median_household_income    0
pct_male                   0
pct_female                 0
pct_less_than_hs           0
pct_bachelors_plus         0
pct_graduate_degree        0
pct_white                  0
pct_black                  0
pct_asian                  0
pct_hispanic               0
dtype: int64


,year,StateAbbr,County name,CountyFIPS,BPHIGH,CASTHMA,COPD,MHLTH,PHLTH,SLEEP,...,median_household_income,pct_male,pct_female,pct_less_than_hs,pct_bachelors_plus,pct_graduate_degree,pct_white,pct_black,pct_asian,pct_hispanic
0,2013,AK,ANCHORAGE MUNICIPALITY,2020,28.691667,NaN,NaN,NaN,NaN,NaN,...,77454.0,50.877431,49.122569,4.749405,13.285259,7.373060,66.225101,5.872909,8.224240,7.984433
1,2013,AL,JEFFERSON COUNTY,1073,41.893750,NaN,NaN,NaN,NaN,NaN,...,45429.0,47.338099,52.661901,8.438817,12.332967,7.788755,53.203847,42.299621,1.474751,3.803496
2,2013,AL,MADISON COUNTY,1089,37.654348,NaN,NaN,NaN,NaN,NaN,...,58434.0,49.020128,50.979872,6.620805,15.974758,9.615096,69.198506,24.052771,2.489986,4.586491
3,2013,AL,MOBILE COUNTY,1097,44.833333,NaN,NaN,NaN,NaN,NaN,...,43028.0,47.976708,52.023292,10.492560,8.919669,4.602989,60.570249,34.773759,1.899862,2.473692
4,2013,AL,MONTGOMERY COUNTY,1101,39.792727,NaN,NaN,NaN,NaN,NaN,...,44790.0,47.572277,52.427723,9.241418,12.436021,7.702419,40.095642,55.189099,2.122893,3.481090


In [15]:
# Check shape and columns match
print(f"Original df shape: {df_original.shape}")
print(f"New df shape:       {df.shape}")

# Check for any new or missing columns
original_cols = set(df_original.columns)
new_cols = set(df.columns)

print(f"\nColumns added:   {new_cols - original_cols}")
print(f"Columns removed: {original_cols - new_cols}")

# Check for any rows lost or gained
print(f"\nRows in original: {len(df_original)}")
print(f"Rows in new df:   {len(df)}")

Original df shape: (6646, 41)
New df shape:       (6646, 41)

Columns added:   set()
Columns removed: set()

Rows in original: 6646
Rows in new df:   6646


In [16]:
df.to_csv("../../data/merged_final_transformed.csv", index=False)
print(f"Saved: {df.shape}")

Saved: (6646, 41)
